NOTE - I tried to have ChatGPT be looser with categorizing elements into multiple attributes.

In [5]:
sgsg_df = pd.read_csv('transformed data/2-Uniqueness and Element avgs.csv')
sgsg_df

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS,AVG_TOOL_UNIQUENESS,AVG_TASK_UNIQUENESS,Mean T3 Uniqueness,Element Avg
0,11-1011.00,Chief Executives,1.285714,1.000,3.290,1.858571,2.442583
1,11-1011.03,Chief Sustainability Officers,1.000000,1.000,3.056,1.685333,2.210463
2,11-1021.00,General and Operations Managers,1.246575,1.278,2.765,1.763192,2.201566
3,11-2011.00,Advertising and Promotions Managers,1.397260,1.000,3.400,1.932420,2.054743
4,11-2021.00,Marketing Managers,1.150327,1.000,3.750,1.966776,2.122046
...,...,...,...,...,...,...,...
882,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.000000,1.647,2.692,1.779667,2.326854
883,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.000000,2.625,2.857,2.160667,2.255412
884,53-7073.00,Wellhead Pumpers,1.125000,2.768,2.714,2.202333,2.192687
885,53-7081.00,Refuse and Recyclable Material Collectors,1.400000,2.812,3.429,2.547000,1.863213


In [6]:
sgsg_df.nlargest(10, 'AVG_TASK_UNIQUENESS')

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS,AVG_TOOL_UNIQUENESS,AVG_TASK_UNIQUENESS,Mean T3 Uniqueness,Element Avg
87,13-2041.00,Credit Analysts,1.634146,1.000,4.182,2.272049,1.845770
618,45-4022.00,Logging Equipment Operators,1.375000,2.632,4.111,2.706000,2.179640
120,15-2021.00,Mathematicians,2.043011,1.143,4.000,2.395337,1.946903
634,47-2072.00,Pile Driver Operators,1.500000,2.400,4.000,2.633333,2.240702
408,29-1291.00,Acupuncturists,1.769231,3.162,3.833,2.921410,1.992862
838,53-1041.00,Aircraft Cargo Handling Supervisors,1.111111,2.625,3.833,2.523037,2.422716
409,29-1292.00,Dental Hygienists,1.210526,2.254,3.812,2.425509,2.124271
100,15-1221.00,Computer and Information Research Scientists,1.736196,2.394,3.800,2.643399,2.112062
440,31-1121.00,Home Health Aides,1.190476,1.469,3.800,2.153159,1.976936
661,47-4021.00,Elevator and Escalator Installers and Repairers,1.375000,1.300,3.800,2.158333,2.404207


In [11]:
df_cognitive_complexity.shape

(894, 93)

In [13]:
df_creativity.shape

(894, 6)

In [14]:
df_data_structure.shape

(894, 12)

In [15]:
df_decision_accountability.shape

(894, 9)

In [16]:
df_physical_requirements.shape

(894, 47)

In [17]:
df_routine_structure.shape

(894, 7)

In [18]:
df_social_interactions.shape

(894, 23)

# V1 - ChatGPT No overlapping
One codeblock

In [10]:
import pandas as pd

df_cognitive_complexity = pd.read_csv('elements sorted/gpt v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/gpt v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/gpt v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/gpt v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/gpt v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/gpt v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/gpt v1 no overlap/social_interactions.csv')

dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]



import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
# avg_dfs["creativity"].head()


avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]


import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)

# Normalize for directionality -
# most of these have a negative impact on automation risk (meaning the score will be lower)
merged_df['physical_requirements_avg'] = 5 - merged_df['physical_requirements_avg']
merged_df['social_interactions_avg'] = 5 - merged_df['social_interactions_avg']
merged_df['cognitive_complexity_avg'] = 5 - merged_df['cognitive_complexity_avg']
merged_df['creativity_avg'] = 5 - merged_df['creativity_avg']
merged_df['decision_accountability_avg'] = 5 - merged_df['decision_accountability_avg']

# Apply weights according to my scale
merged_df['weighted_average_GPT_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

merged_df.to_csv('transformed data-3-GPTv1 non overlap.csv', index=False)

merged_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_GPT_v1_non_overlap
0,11-1011.00,Chief Executives,2.515545,2.543318,2.842733,0.805990,3.518718,2.732,1.524886,2.426642
1,11-1011.03,Chief Sustainability Officers,2.794998,2.502996,2.935123,1.719022,3.525297,2.008,1.897527,2.475137
2,11-1021.00,General and Operations Managers,2.879341,2.996607,2.639833,1.021243,3.413841,2.622,1.992559,2.582860
3,11-2011.00,Advertising and Promotions Managers,3.001268,2.702536,2.481961,1.745224,3.552578,2.522,2.162071,2.639887
4,11-2021.00,Marketing Managers,2.895341,2.738614,2.494049,1.708414,3.602884,2.568,1.948903,2.611707
...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.987875,3.379000,2.500994,0.918124,2.383196,3.274,2.689636,2.712668
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.000908,3.364275,2.218989,1.173527,2.483800,3.152,2.794480,2.716171
891,53-7073.00,Wellhead Pumpers,3.124872,3.504243,2.358553,1.327990,2.462484,2.826,2.760278,2.711636
892,53-7081.00,Refuse and Recyclable Material Collectors,3.478014,3.855050,1.284449,1.996345,2.607217,3.316,3.106563,2.948306


# Original working slow version

In [22]:
import pandas as pd

In [43]:
df_cognitive_complexity = pd.read_csv('elements sorted/gpt v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/gpt v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/gpt v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/gpt v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/gpt v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/gpt v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/gpt v1 no overlap/social_interactions.csv')

In [44]:
dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]

In [25]:
import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
avg_dfs["creativity"].head()

,O*NET-SOC Code,Title,creativity_avg
0,11-1011.00,Chief Executives,2.456682
1,11-1011.03,Chief Sustainability Officers,2.497004
2,11-1021.00,General and Operations Managers,2.003393
3,11-2011.00,Advertising and Promotions Managers,2.297464
4,11-2021.00,Marketing Managers,2.261386


In [34]:
avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]

In [35]:
import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)
merged_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg
0,11-1011.00,Chief Executives,2.484455,2.456682,2.842733,4.194010,1.481282,2.732,3.475114
1,11-1011.03,Chief Sustainability Officers,2.205002,2.497004,2.935123,3.280978,1.474703,2.008,3.102473
2,11-1021.00,General and Operations Managers,2.120659,2.003393,2.639833,3.978757,1.586159,2.622,3.007441
3,11-2011.00,Advertising and Promotions Managers,1.998732,2.297464,2.481961,3.254776,1.447422,2.522,2.837929
4,11-2021.00,Marketing Managers,2.104659,2.261386,2.494049,3.291586,1.397116,2.568,3.051097
...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125,1.621000,2.500994,4.081876,2.616804,3.274,2.310364
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092,1.635725,2.218989,3.826473,2.516200,3.152,2.205520
891,53-7073.00,Wellhead Pumpers,1.875128,1.495757,2.358553,3.672010,2.537516,2.826,2.239722
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986,1.144950,1.284449,3.003655,2.392783,3.316,1.893437


Very dumb version for average. Not signed to account for differences in types.

In [39]:
merged_df['weighted_average_GPT_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

merged_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_GPT_v1_non_overlap
0,11-1011.00,Chief Executives,2.484455,2.456682,2.842733,4.194010,1.481282,2.732,3.475114,2.734704
1,11-1011.03,Chief Sustainability Officers,2.205002,2.497004,2.935123,3.280978,1.474703,2.008,3.102473,2.415087
2,11-1021.00,General and Operations Managers,2.120659,2.003393,2.639833,3.978757,1.586159,2.622,3.007441,2.493907
3,11-2011.00,Advertising and Promotions Managers,1.998732,2.297464,2.481961,3.254776,1.447422,2.522,2.837929,2.365306
4,11-2021.00,Marketing Managers,2.104659,2.261386,2.494049,3.291586,1.397116,2.568,3.051097,2.414302
...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125,1.621000,2.500994,4.081876,2.616804,3.274,2.310364,2.597131
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092,1.635725,2.218989,3.826473,2.516200,3.152,2.205520,2.488427
891,53-7073.00,Wellhead Pumpers,1.875128,1.495757,2.358553,3.672010,2.537516,2.826,2.239722,2.390475
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986,1.144950,1.284449,3.003655,2.392783,3.316,1.893437,2.134984


In [ ]:
Routine - 20%

Cognitive complexity - 15%

Physical requirements - 15%

Social interactions - 15%

Creativity - 15%

Decision accountability 10%

Data structure 10%

In [28]:
df_cognitive_complexity_avg

,O*NET-SOC Code,Title,cognitive_complexity_avg
0,11-1011.00,Chief Executives,2.484455
1,11-1011.03,Chief Sustainability Officers,2.205002
2,11-1021.00,General and Operations Managers,2.120659
3,11-2011.00,Advertising and Promotions Managers,1.998732
4,11-2021.00,Marketing Managers,2.104659
...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092
891,53-7073.00,Wellhead Pumpers,1.875128
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986


In [30]:
df_creativity_avg

,O*NET-SOC Code,Title,creativity_avg
0,11-1011.00,Chief Executives,2.456682
1,11-1011.03,Chief Sustainability Officers,2.497004
2,11-1021.00,General and Operations Managers,2.003393
3,11-2011.00,Advertising and Promotions Managers,2.297464
4,11-2021.00,Marketing Managers,2.261386
...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.621000
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.635725
891,53-7073.00,Wellhead Pumpers,1.495757
892,53-7081.00,Refuse and Recyclable Material Collectors,1.144950


# With overlaps
For this run, I permitted ChatGPT to assign an element into multiple categories, which will result in some being weighed twice. I decided against this initially but have come to consider that maybe it *should* matter more.